In [1]:
import ast
import json
import os
import uuid

import numpy as np
import pandas as pd

from datasets import load_dataset
from fastembed import SparseTextEmbedding
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer

/home/chirag/Documents/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# MS MARCO Dataset Preparation: Download, Parquet, and JSON Export

In [3]:
DATA_LIMIT = 2000

HF_DATASET = "microsoft/ms_marco"
HF_CONFIG = "v1.1"

DATA_DIR = "data"

PARQUET_FILE = os.path.join(DATA_DIR,"ms_marco_validation.parquet")
JSON_FILE = os.path.join(DATA_DIR,"ms_marco_top2k.json")


In [6]:

# create data folder if not exists
os.makedirs(DATA_DIR, exist_ok=True)

In [7]:
# -----------------------------
# 1. Load validation split
# -----------------------------

print("Loading validation dataset from Hugging Face...")

dataset = load_dataset(
    HF_DATASET,
    HF_CONFIG,
    split="validation"
)

print(f"Validation rows loaded: {len(dataset)}")


# -----------------------------
# 2. Convert to Pandas
# -----------------------------

df = dataset.to_pandas()

print(f"DataFrame shape: {df.shape}")


# -----------------------------
# 3. Save validation split as Parquet
# -----------------------------

df.to_parquet(
    PARQUET_FILE,
    index=False
)

print(f"Saved validation dataset to: {PARQUET_FILE}")

Loading validation dataset from Hugging Face...


Generating test split: 100%|██████████| 9650/9650 [00:00<00:00, 78362.23 examples/s]


Validation rows loaded: 10047
DataFrame shape: (10047, 6)
Saved validation dataset to: data/ms_marco_validation.parquet


In [8]:
# -----------------------------
# Recursively convert to JSON-safe Python objects
# -----------------------------

def make_json_serializable(obj):

    # NumPy array
    if isinstance(obj, np.ndarray):
        return [
            make_json_serializable(x)
            for x in obj.tolist()
        ]

    # NumPy scalar
    if isinstance(obj, np.generic):
        return obj.item()

    # Dictionary
    if isinstance(obj, dict):
        return {
            str(k): make_json_serializable(v)
            for k, v in obj.items()
        }

    # List / tuple
    if isinstance(obj, (list, tuple)):
        return [
            make_json_serializable(x)
            for x in obj
        ]

    # Pandas NA / NaN
    if obj is None:
        return None

    if pd.isna(obj):
        return None

    return obj



# -----------------------------
# 4. Read the Parquet file
# -----------------------------

df = pd.read_parquet(
    PARQUET_FILE
)

print(f"Read Parquet file: {df.shape}")


# -----------------------------
# 5. Take first DATA_LIMIT records
# -----------------------------

records = df.head(DATA_LIMIT).to_dict(
    orient="records"
)


# -----------------------------
# 6. Convert everything to JSON-safe objects
# -----------------------------

records = make_json_serializable(records)


# -----------------------------
# 7. Create final JSON structure
# -----------------------------

ms_marco_data = {
    "dataset_name": "ms_marco",
    "dataset_link":
        "https://huggingface.co/datasets/microsoft/ms_marco",
    "split": "validation",
    "total_documents": len(records),
    "documents": records
}


# -----------------------------
# 8. Save JSON
# -----------------------------

with open(
    JSON_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        ms_marco_data,
        f,
        indent=2,
        ensure_ascii=False,
        allow_nan=False
    )


print(
    f"Created {JSON_FILE} "
    f"with {len(records)} documents"
)

Read Parquet file: (10047, 6)
Created data/ms_marco_top2k.json with 2000 documents


# MS MARCO — Qdrant Database Setup and Data Ingestion

In [4]:
# Load the JSON file
with open(JSON_FILE, "r") as f:
    documents = json.load(f)


# The actual documents are stored inside the "documents" key
documents = documents["documents"]

print(f"Loaded {len(documents)} documents")

Loaded 2000 documents


In [5]:
# ============================================================
# 2. Qdrant Configuration
# ============================================================

# Name of the Qdrant collection
COLLECTION_NAME = "ms_marco_hybrid"


# Connect to the local Qdrant instance
client = QdrantClient(
    url="http://localhost:6333"
)

In [6]:
# ============================================================
# 3. Initialize Embedding Models
# ============================================================

# ------------------------------------------------------------
# Dense Embedding Model
# ------------------------------------------------------------
# This model converts text into a dense numerical vector.
# We use MiniLM because it is lightweight and suitable
# for semantic similarity search.
# ------------------------------------------------------------

dense_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)


# ------------------------------------------------------------
# Sparse Embedding Model
# ------------------------------------------------------------
# BM25 is used as the sparse retrieval component.
# Sparse embeddings are useful for exact keyword-based
# matching and complement dense semantic search.
# ------------------------------------------------------------

sparse_model = SparseTextEmbedding(
    model_name="Qdrant/bm25"
)


# Get the dimensionality of the dense embedding model.
DENSE_DIM = dense_model.get_sentence_embedding_dimension()
print("Dense vector dimension:", DENSE_DIM)


Fetching 18 files: 100%|██████████| 18/18 [00:00<00:00, 18.30it/s]

Dense vector dimension: 384



/tmp/ipykernel_77047/1253258163.py:32: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  DENSE_DIM = dense_model.get_sentence_embedding_dimension()


In [7]:
# ============================================================
# 4. Create Qdrant Collection
# ============================================================

def create_collection():
    """
    Create the Qdrant collection used for hybrid search.

    The collection contains:
    - Dense vectors for semantic search
    - Sparse vectors for BM25 keyword-based search

    If the collection already exists, it is deleted first
    so that we can create a fresh collection.
    """

    # --------------------------------------------------------
    # Delete the collection if it already exists
    # --------------------------------------------------------

    if client.collection_exists(COLLECTION_NAME):
        client.delete_collection(COLLECTION_NAME)

        print(f"Deleted existing collection: {COLLECTION_NAME}")


    # --------------------------------------------------------
    # Create a new collection
    # --------------------------------------------------------

    client.create_collection(
        collection_name=COLLECTION_NAME,

        # Dense vector configuration
        vectors_config={
            "dense": models.VectorParams(
                size=DENSE_DIM,
                distance=models.Distance.COSINE
            )
        },

        # Sparse vector configuration
        sparse_vectors_config={
            "sparse": models.SparseVectorParams(
                modifier=models.Modifier.IDF
            )
        }
    )

    print(f"Created collection: {COLLECTION_NAME}")


# Create the collection
create_collection()


Created collection: ms_marco_hybrid


In [8]:
# ============================================================
# 5. Helper Function for Parsing List Values
# ============================================================

def parse_list_string(value):
    """
    Convert a value into a Python list.

    Some fields in the dataset may already be stored as lists,
    while others may be stored as strings representing lists.

    Example:
        "['text 1', 'text 2']"
            ->
        ['text 1', 'text 2']
    """

    # If the value is already a list, return it directly.
    if isinstance(value, list):
        return value

    # Otherwise, safely convert the string representation
    # into a Python object.
    return ast.literal_eval(value)



In [9]:
# ============================================================
# 6. Create Qdrant Points
# ============================================================

def create_points(documents):
    """
    Convert the dataset documents into Qdrant points.

    Each passage is converted into:
        1. A dense embedding
        2. A sparse BM25 embedding
        3. Metadata stored in the payload

    Returns:
        A list of Qdrant PointStruct objects.
    """

    points = []


    # --------------------------------------------------------
    # Process each document
    # --------------------------------------------------------

    for document in documents:

        # Unique ID associated with the original query
        query_id = document["query_id"]


        # ----------------------------------------------------
        # Extract passages and URLs
        # ----------------------------------------------------

        passages = parse_list_string(
            document["passages"]["passage_text"]
        )

        urls = parse_list_string(
            document["passages"]["url"]
        )


        # Original search query
        query = document["query"]


        # ----------------------------------------------------
        # Process each passage/chunk
        # ----------------------------------------------------

        for chunk_idx, text in enumerate(passages):

            # Create a readable ID for the chunk
            chunk_id = f"{query_id}_{chunk_idx}"


            # =================================================
            # Dense Embedding
            # =================================================
            # Convert the passage into a dense semantic vector.
            #
            # normalize_embeddings=True normalizes the vector,
            # which works well with cosine similarity.
            # =================================================

            dense_vector = dense_model.encode(
                text,
                normalize_embeddings=True
            ).tolist()


            # =================================================
            # Sparse BM25 Embedding
            # =================================================
            # Generate a sparse representation of the passage
            # using the BM25 model.
            # =================================================

            sparse_vector = list(
                sparse_model.embed([text])
            )[0]


            # Convert the FastEmbed sparse vector into
            # Qdrant's SparseVector format.
            sparse = models.SparseVector(
                indices=sparse_vector.indices.tolist(),
                values=sparse_vector.values.tolist()
            )


            # =================================================
            # Metadata / Payload
            # =================================================
            # Store useful information along with the vectors.
            # This metadata can later be returned during search.
            # =================================================

            payload = {
                "query_id": query_id,
                "chunk_id": chunk_id,
                "query": query,
                "text": text,
                "answer": document["answers"],
                "url": urls[chunk_idx],

                # Optional fields
                "query_type": document.get("query_type"),
                "wellFormedAnswers": document.get(
                    "wellFormedAnswers"
                )
            }


            # =================================================
            # Create Qdrant Point
            # =================================================

            points.append(
                models.PointStruct(
                    # Generate a unique ID for this point
                    id=str(uuid.uuid4()),

                    # Store both dense and sparse vectors
                    vector={
                        "dense": dense_vector,
                        "sparse": sparse
                    },

                    # Store the metadata
                    payload=payload
                )
            )


    return points


In [10]:
# ============================================================
# 7. Generate All Qdrant Points
# ============================================================

points = create_points(documents)

print("Total points:", len(points))

Total points: 16396


In [11]:
# ============================================================
# 8. Upload Points to Qdrant in Batches
# ============================================================

# Uploading in batches avoids sending the entire dataset
# to Qdrant in a single request.
BATCH_SIZE = 100


for i in range(0, len(points), BATCH_SIZE):

    # Select the current batch
    batch = points[i:i + BATCH_SIZE]


    # --------------------------------------------------------
    # Upload the batch to Qdrant
    # --------------------------------------------------------

    client.upsert(
        collection_name=COLLECTION_NAME,
        points=batch,
        wait=True
    )


    # --------------------------------------------------------
    # Display upload progress
    # --------------------------------------------------------

    uploaded = min(
        i + BATCH_SIZE,
        len(points)
    )

    print(
        f"Uploaded {uploaded}/{len(points)}"
    )


# ============================================================
# 9. Finished
# ============================================================

print(
    f"\nSuccessfully uploaded {len(points)} points "
    f"to collection '{COLLECTION_NAME}'."
)

Uploaded 100/16396
Uploaded 200/16396
Uploaded 300/16396
Uploaded 400/16396
Uploaded 500/16396
Uploaded 600/16396
Uploaded 700/16396
Uploaded 800/16396
Uploaded 900/16396
Uploaded 1000/16396
Uploaded 1100/16396
Uploaded 1200/16396
Uploaded 1300/16396
Uploaded 1400/16396
Uploaded 1500/16396
Uploaded 1600/16396
Uploaded 1700/16396
Uploaded 1800/16396
Uploaded 1900/16396
Uploaded 2000/16396
Uploaded 2100/16396
Uploaded 2200/16396
Uploaded 2300/16396
Uploaded 2400/16396
Uploaded 2500/16396
Uploaded 2600/16396
Uploaded 2700/16396
Uploaded 2800/16396
Uploaded 2900/16396
Uploaded 3000/16396
Uploaded 3100/16396
Uploaded 3200/16396
Uploaded 3300/16396
Uploaded 3400/16396
Uploaded 3500/16396
Uploaded 3600/16396
Uploaded 3700/16396
Uploaded 3800/16396
Uploaded 3900/16396
Uploaded 4000/16396
Uploaded 4100/16396
Uploaded 4200/16396
Uploaded 4300/16396
Uploaded 4400/16396
Uploaded 4500/16396
Uploaded 4600/16396
Uploaded 4700/16396
Uploaded 4800/16396
Uploaded 4900/16396
Uploaded 5000/16396
Uploaded 